# Problem 3 — Phase 4: Kernel pilot (not full tree)

**Index:** [`hierarchical_problem3_index.ipynb`](hierarchical_problem3_index.ipynb) · **Prev:** [Phase 3](hierarchical_problem3_phase3_by_depth.ipynb)

**Main pilot:** sample **20 random binary edges** (same eligibility as Phase 3: train has two classes, subtree-restricted pool). For each edge, subsample **train** for fitting, then score **F1 on the full test** split; report **macro** (mean over edges) and **pooled micro** F1 over all edge×test-doc pairs.

[`pilot_kernel_compare_on_edges`](hierarchical_summary_metrics.py) compares:

- **Linear** — `TfidfVectorizer` + `LinearSVC` (same spirit as the full-tree linear router).
- **RBF** — TF-IDF + `SVC` (RBF kernel).
- **Explicit degree-2** — TF-IDF → `TruncatedSVD` → `PolynomialFeatures(degree=2)` → `LinearSVC` (squares and pairwise interactions of SVD components; cheap explicit nonlinear features).

Optional single-edge helper: [`pilot_kernel_svc_on_edge`](hierarchical_summary_metrics.py) (internal train/val split, `SVC` linear / RBF / built-in poly kernel on one edge).


In [ ]:
from __future__ import annotations

import json
import random
from pathlib import Path

import pandas as pd
from IPython.display import display

from hierarchical_summary_metrics import pilot_kernel_compare_on_edges, pilot_kernel_svc_on_edge
from hierarchical_training_data import make_multilabel_binary_pool_data
from topic_hierarchy import BinaryEdgeSpec, TopicTree, binary_edge_specs, load_topic_tree, summary


def homework4_base() -> Path:
    here = Path.cwd().resolve()
    if (here / "topics.csv").exists():
        return here
    for p in [here, *here.parents]:
        if (p / "topics.csv").exists():
            return p
    raise FileNotFoundError("topics.csv not found — cwd should be Homework 4")


BASE = homework4_base()
tree = load_topic_tree(str(BASE / "topics.csv"))
mldata = make_multilabel_binary_pool_data(base_path=str(BASE))
RESTRICT = True
MAX_FEATURES = 8000

N_PILOT_EDGES = 20
PILOT_SEED = 42
MAX_TRAIN_SUBSAMPLE = 3000
SVD_COMPONENTS = 100

print("BASE", BASE)
print(summary(tree))
print("train", len(mldata.train_ids()), "test", len(mldata.test_ids()))


In [ ]:
root = tree.traversal_root
children = tree.children.get(root, [])
pilot_edge = None
for ch in children:
    Xtr, ytr = mldata.binary_edge_dataset(root, ch, "train")
    if len(set(ytr)) >= 2 and len(Xtr) >= 100:
        pilot_edge = (root, ch)
        break
if pilot_edge:
    print("Pilot edge", pilot_edge)
    pilot_res = pilot_kernel_svc_on_edge(mldata, pilot_edge[0], pilot_edge[1])
    display(pd.DataFrame(pilot_res).T)
else:
    print("No suitable H1 edge for pilot")


### Notes

If linear ≈ nonlinear F1 but kernels are much slower per fit, prefer **linear `LinearSVC`** for full-tree training. Full-tree kernel SVMs are usually impractical here.
